# Generate text embeddings for question data

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import col_config_sensitivity

from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 500)
from shutil import rmtree

Load data

In [ ]:
## Processed data
data_dir = BASE_PATH / "data" / "cleaned" / "processed"
self_df = pd.read_parquet(data_dir / "self.parquet")
parent_df = pd.read_parquet(data_dir / "parent.parquet")
## Meta data
meta_dir = BASE_PATH / "data" / "cleaned" / "meta_text"
meta_self = pd.read_csv(meta_dir / "self.csv", index_col=0)
meta_parent = pd.read_csv(meta_dir / "parent.csv", index_col=0)
## Columns
self_impact_cols = col_config_sensitivity["self_impact"]
parent_impact_cols = col_config_sensitivity["parent_impact"]
self_text_cols = col_config_sensitivity["self_text"]
parent_text_cols = col_config_sensitivity["parent_text"]
self_quest_cols = col_config_sensitivity["self_quest"]
parent_quest_cols = col_config_sensitivity["parent_quest"]
meta_cols = col_config_sensitivity["meta"]

## Concatenate Text

In [ ]:
from src.embed import concat_question_answer

qa_concat_self = concat_question_answer(answer_df=self_df, question_df=meta_self)
qa_concat_parent = concat_question_answer(answer_df=parent_df, question_df=meta_parent)

Export

In [ ]:
export_dir = BASE_PATH / "data" / "cleaned" / "question_answer"
if export_dir.exists():
    rmtree(export_dir)
export_dir.mkdir(exist_ok=False, parents=True)
qa_concat_self.to_parquet(export_dir / "self.parquet")
qa_concat_parent.to_parquet(export_dir / "parent.parquet")

## Get Embeddings

Choose model

In [ ]:
## MODEL
model_dir = Path("/lscratch3/HuggingFace")
models = []
for idx, entry in enumerate(model_dir.iterdir()):
    print(f"{idx} \t {entry.name}")
    models.append(entry.name)

In [ ]:
MODEL_ID = model_dir / models[12]
print(f"Using model: {str(MODEL_ID.name)}")
assert MODEL_ID.exists()

Run embedding

In [ ]:
import subprocess
import signal


def make_cmd(model_id, impact_type_list, text_col_name):
    model_name = model_id.name
    og_df_dir = BASE_PATH / "data" / "cleaned" / "question_answer"
    result_dir = BASE_PATH / "data" / "w_embeddings"
    log_dir = BASE_PATH / "logs" / "embed" / model_name
    if "70B" in model_name.upper():
        gpu_blocks = ["0,1,2,3", "4,5,6,7"]
    else:
        gpu_blocks = ["0,1", "2,3", "4,5", "6,7"]
    cmd = [
        "python",
        "src/embed_controller.py",
        "--gpu-groups",
        *gpu_blocks,
        "--impact-types",
        *impact_type_list,
        "--model-id",
        str(model_id),
        "--og-imp-dir",
        str(og_df_dir),
        "--text-col-name",
        text_col_name,
        "--log-dir",
        str(log_dir),
        "--result-dir",
        str(result_dir),
    ]
    return cmd

In [ ]:
IMPACT_TYPES = ["self", "parent"]
cmd = make_cmd(
    model_id=MODEL_ID,
    impact_type_list=IMPACT_TYPES,
    # will compute for all text cols if "ALL" passed in, otherwise just single column
    text_col_name="ALL",
)
env = os.environ.copy()
env["PYTHONPATH"] = str(BASE_PATH)
proc = subprocess.Popen(
    cmd,
    cwd=BASE_PATH,
    start_new_session=True,
    env=env,
)
print("Controller PID:", proc.pid)

Monitor

In [ ]:
proc.poll()

Gentle quit

In [ ]:
os.killpg(proc.pid, signal.SIGTERM)

Force kill

In [ ]:
os.killpg(proc.pid, signal.SIGKILL)

## Check results

In [ ]:
import_dir = BASE_PATH / "data" / "w_embeddings"
model_name = MODEL_ID.name
parent_df = pd.read_parquet(import_dir / model_name / "parent.parquet")
self_df = pd.read_parquet(import_dir / model_name / "self.parquet")

In [ ]:
parent_df.columns.to_list()